<a href="https://colab.research.google.com/github/juzt1n/pawscan/blob/main/TempConvNext92_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install --upgrade protobuf

In [ ]:
pip install -q -U keras-tuner

In [ ]:
pip install importlib_resources

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np, json
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import gc
import requests
from PIL import Image
from io import BytesIO
from tensorflow.keras.callbacks import ModelCheckpoint
from google.colab import files
import keras_tuner as kt
import matplotlib.patches as patches
import shutil
import os
import json

In [ ]:
(ds_train, ds_val), ds_info = tfds.load(
    'stanford_dogs',
    split=['train', 'test'],
    as_supervised=False,
    with_info=True
)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

...

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/stanford_dogs/incomplete.336GNC_0.2.0/stanford_dogs-train.tfrecord-[0-9][0…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/stanford_dogs/incomplete.336GNC_0.2.0/stanford_dogs-test.tfrecord-[0-9][0-…

Dataset stanford_dogs downloaded and prepared to /root/tensorflow_datasets/stanford_dogs/0.2.0. Subsequent calls will reuse this data.


In [ ]:
num_classes = ds_info.features['label'].num_classes
num_train_examples = ds_info.splits['train'].num_examples
num_val_examples = ds_info.splits['test'].num_examples
class_names = ds_info.features['label'].names

print("\n--- Dataset Info ---")
print(f"Number of classes: {num_classes}")
print(f"Training examples: {num_train_examples}")
print(f"Validation examples: {num_val_examples}")


--- Dataset Info ---
Number of classes: 120
Training examples: 12000
Validation examples: 8580


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2)
])

def preprocess_image_with_bbox(example, img_size=IMG_SIZE):
    image = example['image']
    label = example['label']
    bboxes = example['objects']['bbox']

    if tf.size(bboxes) > 0:
        bbox = bboxes[0]
        ymin_norm, xmin_norm, ymax_norm, xmax_norm = tf.unstack(bbox)

        img_shape = tf.shape(image)
        height = tf.cast(img_shape[0], tf.float32)
        width = tf.cast(img_shape[1], tf.float32)

        ymin_pixel = tf.cast(tf.round(ymin_norm * height), tf.int32)
        xmin_pixel = tf.cast(tf.round(xmin_norm * width), tf.int32)
        ymax_pixel = tf.cast(tf.round(ymax_norm * height), tf.int32)
        xmax_pixel = tf.cast(tf.round(xmax_norm * width), tf.int32)

        crop_height = tf.maximum(1, ymax_pixel - ymin_pixel)
        crop_width = tf.maximum(1, xmax_pixel - xmin_pixel)

        ymin_pixel = tf.clip_by_value(ymin_pixel, 0, tf.cast(height, tf.int32) - crop_height)
        xmin_pixel = tf.clip_by_value(xmin_pixel, 0, tf.cast(width, tf.int32) - crop_width)

        cropped_image = tf.image.crop_to_bounding_box(
            image,
            offset_height=ymin_pixel,
            offset_width=xmin_pixel,
            target_height=crop_height,
            target_width=crop_width
        )
    else:
        cropped_image = image

    processed_image = tf.image.resize(cropped_image, (img_size, img_size))

    return processed_image, label

ds_train_processed = (
    ds_train.map(lambda x: preprocess_image_with_bbox(x, IMG_SIZE), num_parallel_calls=AUTOTUNE) # Map the raw example dict
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(buffer_size=AUTOTUNE)
)

ds_val_processed = (
    ds_val.map(lambda x: preprocess_image_with_bbox(x, IMG_SIZE), num_parallel_calls=AUTOTUNE) # Map the raw example dict
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=AUTOTUNE)
)

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in ds_val_processed], axis=0)

In [ ]:
def build_comparison_model(model_name, num_classes):
    inputs = tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    if model_name == "convnext_tiny":
        x = tf.keras.applications.convnext.preprocess_input(inputs)
        base = tf.keras.applications.ConvNeXtTiny(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
    else:
        raise ValueError(f"Unknown target: {model_name}")

    base.trainable = False
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
model_targets = ["convnext_tiny"]
evaluation_results = {}

for target in model_targets:
    print(f"\n=========================================")
    print(f"RUNNING: {target.upper()}")
    print(f"=========================================")

    model = build_comparison_model(target, num_classes=num_classes)

    checkpoint_filepath = f'best_{target}.weights.h5'
    model_checkpoint_callback = ModelCheckpoint(
        filepath=checkpoint_filepath,
        save_weights_only=True,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True
    )

    history = model.fit(ds_train_processed, validation_data=ds_val_processed, epochs=3, callbacks=[model_checkpoint_callback])
    print(f"Saved best weights for {target} to {checkpoint_filepath}")

    print(f"Evaluating metrics for {target}...")
    model.load_weights(checkpoint_filepath)
    y_pred_probs = model.predict(ds_val_processed)
    y_pred = np.argmax(y_pred_probs, axis=1)

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    evaluation_results[target] = {
        'accuracy': report['accuracy'],
        'precision': report['macro avg']['precision'],
        'recall': report['macro avg']['recall'],
        'f1-score': report['macro avg']['f1-score'],
        'confusion_matrix': cm
    }

    del model
    tf.keras.backend.clear_session()
    gc.collect()

print("\nAll models trained and safely completed!")


RUNNING: CONVNEXT_TINY
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/3
375/375 ━━━━━━━━━━━━━━━━━━━━ 230s 564ms/step - accuracy: 0.6593 - loss: 1.8169 - val_accuracy: 0.9154 - val_loss: 0.3157
Epoch 2/3
375/375 ━━━━━━━━━━━━━━━━━━━━ 210s 555ms/step - accuracy: 0.8031 - loss: 0.7603 - val_accuracy: 0.9223 - val_loss: 0.2626
Epoch 3/3
375/375 ━━━━━━━━━━━━━━━━━━━━ 262s 553ms/step - accuracy: 0.8242 - loss: 0.6391 - val_accuracy: 0.9242 - val_loss: 0.2549
Saved best weights for convnext_tiny to best_convnext_tiny.weights.h5
Evaluating metrics for convnext_tiny...
269/269 ━━━━━━━━━━━━━━━━━━━━ 45s 149ms/step

All models trained and safely completed!


In [ ]:
CHOSEN = "convnext_tiny"   # <- put the winner from your comparison table here

tf.keras.backend.clear_session()
model = build_comparison_model(CHOSEN, num_classes=num_classes)

checkpoint_filepath = f'best_{CHOSEN}.weights.h5'
model.load_weights(checkpoint_filepath)

model.save("pawscan_dog_model.keras")

with open("class_names.json", "w") as f:
    json.dump(list(class_names), f)

print("Saved model + class names with best weights.")

Saved model + class names with best weights.


In [ ]:
model = tf.keras.models.load_model("pawscan_dog_model.keras")
try:
    class_names
except NameError:
    class_names = json.load(open("class_names.json"))

THRESHOLD = 0.40       # top score must clear this to report a breed; tune this

def clean(name):                       # 'n02085620-chihuahua' -> 'chihuahua'
    return name.split('-', 1)[-1].replace('_', ' ') if '-' in name else name

def preprocess_for_prediction(pil_img, img_size=IMG_SIZE):
    tf_img = tf.convert_to_tensor(np.array(pil_img))

    dummy_example = {
        'image': tf_img,
        'label': tf.constant(0, dtype=tf.int64),
        'objects': {'bbox': tf.constant([], dtype=tf.float32)}
    }

    processed_image_tensor, _ = preprocess_image_with_bbox(dummy_example, img_size)

    return np.expand_dims(processed_image_tensor.numpy(), axis=0)

uploaded = files.upload()      # pick one or more images

for fname in uploaded:
    img = Image.open(fname)
    preds = model.predict(preprocess_for_prediction(img), verbose=0)[0]
    top5 = preds.argsort()[-5:][::-1]
    top_conf = preds[top5[0]]

    plt.figure(figsize=(4, 4))
    plt.imshow(img); plt.axis("off")
    if top_conf < THRESHOLD:
        plt.title(f"No confident dog match ({top_conf*100:.1f}%) -- Try again with another picture!")
    else:
        plt.title(f"{clean(class_names[top5[0]])}  ({top_conf*100:.1f}%)")
    plt.show()

    print(f"Results for {fname}:")
    if top_conf < THRESHOLD:
        print(f"  ⚠ Below threshold ({THRESHOLD*100:.0f}%) — likely not a dog or unclear photo.")
    print("  Top 5 raw scores:")
    for i in top5:
        print(f"    {clean(class_names[i]):<28} {preds[i]*100:5.1f}%")
    print("-" * 45)

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 6 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
